<a href="https://colab.research.google.com/github/UniVR-DH/DKR-course/blob/main/L18-advanced/Grafeo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grafeo

An example of native embeddable Graph DBMS [Grafeo](https://pypi.org/project/grafeo/)

**Query Processing¶**

    Parser - GQL/Cypher/SPARQL/Gremlin/GraphQL/SQL-PGQ to AST
    Binder - Semantic analysis and type checking
    Planner - AST to logical plan
    Optimizer - Cost-based optimization
    Executor - Push-based execution




In [1]:
%pip install grafeo

In [2]:
# Initialize local store
from grafeo import GrafeoDB

# In-memory database
db = GrafeoDB()

# Or persistent on file
# db = GrafeoDB("./my-graph")

In [3]:
# Create nodes
db.execute(("CREATE GRAPH my_graph"))
db.execute(("USE GRAPH my_graph"))
db.execute("INSERT (:Person {uid:10, name: 'Alix', age: 30})")
db.execute("INSERT (:Person {uid:20, name: 'Gus', age: 25})")
result = db.execute("""
MATCH (a:Person {uid: 10}), (b:Person {uid: 20})
INSERT (a)-[:KNOWS {since: 2020, project: 'Alpha'}]->(b)
""")
print(result)

(empty)


In [4]:
# Find edges
result = db.execute("""
    MATCH (a:Person)-[r:KNOWS]->(b:Person)
    RETURN a.name AS from, b.name AS to, r.since
""")

for row in result:
  print(f"{row['from']} knows {row['to']} since {row['r.since']}")


Alix knows Gus since 2020


In [5]:
db.execute(("DROP GRAPH IF EXISTS my_graph "))

QueryResult(columns=[], rows=0)

In [8]:
import pandas as pd

PERSON_URL = "https://gist.githubusercontent.com/Dtenwolde/2b02aebbed3c9638a06fda8ee0088a36/raw/8c4dc551f7344b12eaff2d1438c9da08649d00ec/person-sf0.003.csv"
KNOWS_URL = "https://gist.githubusercontent.com/Dtenwolde/81c32c9002d4059c2c3073dbca155275/raw/8b440e810a48dcaa08c07086e493ec0e2ec6b3cb/person_knows_person-sf0.003.csv"


person_df = pd.read_csv(PERSON_URL)
knows_df = pd.read_csv(KNOWS_URL)

person_df[:14]

,creationDate,id,firstName,lastName,gender,birthday,locationIP,browserUsed,LocationCityId,speaks,email
0,2010-01-03 23:10:31.499+00,14,Hossein,Forouhar,male,1984-03-11,77.245.239.11,Firefox,1166,fa;ku;en,Hossein14@hotmail.com
1,2010-01-31 21:13:03.929+00,16,Jan,Zakrzewski,female,1986-07-05,31.41.169.140,Chrome,1284,pl;en,Jan16@hotmail.com;Jan16@gmx.com;Jan16@gmail.co...
2,2010-02-13 06:05:24.513+00,32,Miguel,Gonzalez,male,1981-09-17,148.204.226.31,Chrome,737,es;en,Miguel32@gmx.com;Miguel32@gmail.com;Miguel32@h...
3,2010-03-25 01:14:04.882+00,2199023255557,Eric,Mettacara,male,1989-08-05,203.215.63.48,Firefox,1014,my;en,Eric2199023255557@gmx.com;Eric2199023255557@gm...
4,2010-04-18 08:27:21.494+00,2199023255573,Arbaaz,Ali,female,1987-01-10,115.186.113.189,Safari,779,ur;en,Arbaaz2199023255573@gmail.com;Arbaaz2199023255...
5,2010-03-21 19:25:42.685+00,2199023255594,Ali,Achiou,female,1981-03-11,196.29.42.107,Firefox,966,ar;fr;en,Ali2199023255594@poop.com;Ali2199023255594@mai...
6,2010-06-15 17:00:57.698+00,4398046511139,Ayesha,Ahmed,male,1988-11-12,202.4.167.190,Chrome,786,ur;en,Ayesha4398046511139@gmx.com;Ayesha439804651113...
7,2010-08-08 08:41:16.348+00,6597069766702,Alejandro,Garcia,male,1986-12-16,189.202.102.202,Firefox,750,es;en,Alejandro6597069766702@hotmail.com;Alejandro65...
8,2010-09-18 04:25:01.182+00,8796093022234,Rahul,Sharma,female,1985-11-12,49.50.77.195,Firefox,125,mr;kn;en,Rahul8796093022234@gmail.com
9,2010-10-28 12:49:29.47+00,8796093022237,Lei,Zhang,male,1986-07-23,1.2.2.77,Firefox,452,zh;en,Lei8796093022237@gmail.com;Lei8796093022237@gm...


In [9]:
knows_df[:14]

,creationDate,Person1Id,Person2Id
0,2012-10-07 02:24:40.381+00,14,10995116277782
1,2012-07-08 15:27:12.264+00,14,24189255811081
2,2012-11-26 06:45:21.004+00,14,26388279066668
3,2011-11-08 06:05:10.543+00,16,2199023255594
4,2012-06-14 12:43:25.817+00,16,26388279066655
5,2012-09-22 23:53:26.452+00,16,28587302322180
6,2012-06-30 00:45:34.905+00,16,28587302322204
7,2011-06-24 09:40:20.246+00,32,2199023255594
8,2012-08-18 11:04:48.36+00,32,13194139533352
9,2012-11-12 10:57:04.309+00,32,17592186044461


In [10]:
db.execute(("CREATE GRAPH IF NOT EXISTS social"))
db.execute(("USE GRAPH social"))
count_nodes = 0
for _, p in person_df.iterrows():
    speaks_list = str(p["speaks"]).split(";") if pd.notna(p["speaks"]) else []
    email_list = str(p["email"]).split(";") if pd.notna(p["email"]) else []
    count_nodes += 1
    result = db.execute(f"""
        INSERT (:Person {{id: {p["id"]%50},
            firstName: '{p["firstName"]}',
            lastName: '{p["lastName"]}',
            gender: '{p["gender"]}',
            creationDate: '{p["creationDate"]}',
            locationIP: '{p["locationIP"]}',
            browserUsed: '{p["browserUsed"]}',
            locationCityId: {p["LocationCityId"]},
            speaks: {speaks_list},
            email: {email_list}
        }})
    """)

count_edges = 0
for _, k in knows_df.iterrows():
    result = db.execute(f"""
        MATCH (p1:Person {{id: {k["Person1Id"]%50}}}), (p2:Person {{id: {k["Person2Id"]%100}}})
        INSERT (p1)
        -[:KNOWS {{creationDate: '{k["creationDate"]}'}}]->
        (:p2)
    """)
    count_edges += 1

print(f"Loaded {count_nodes} nodes and {count_edges} edges")

Loaded 50 nodes and 83 edges


In [11]:
# Query the graph
result = db.execute("""
MATCH (p:Person)
RETURN count(p) AS total_nodes;
""")
for row in result:
    print(row)

result = db.execute("""
MATCH ()-[r:KNOWS]->()
RETURN count(r) AS total_edges;
""")
for row in result:
    print(row)

{'total_nodes': 50}
{'total_edges': 75}


In [18]:
# Query the graph
result = db.execute("""
MATCH (p:Person)
WHERE p.locationCityId = 255
RETURN p.id, p.firstName, p.lastName, p.speaks
""")
for row in result:
    print(row)

{'p.id': 44, 'p.firstName': 'John', 'p.lastName': 'Reddy', 'p.speaks': ['ml', 'bn', 'en']}


In [19]:
# Find connections
result = db.execute("""
MATCH (p1:Person) -[:KNOWS]- {1,4} (p2:Person)
WHERE p1.id <> p2.id
RETURN p1.firstName, p1.lastName, p2.firstName, p2.lastName
""")
for row in result:
    print(row)

## Grafeo also supports RDF in the same store

In [20]:
result = db.execute_sparql("""
PREFIX : <http://example.org/>
INSERT DATA {
    :Alix a :Person ;
           :name "Alix" ;
           :age 30 .
}
""")


In [21]:
result = db.execute_sparql("SELECT * WHERE { ?s ?p ?o}")               # SPARQL
for row in result:
    print(row)

{'s': 'http://example.org/Alix', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://example.org/Person'}
{'s': 'http://example.org/Alix', 'p': 'http://example.org/name', 'o': 'Alix'}
{'s': 'http://example.org/Alix', 'p': 'http://example.org/age', 'o': '30'}
